In [1]:
import pandas as pd
import joblib

df = pd.read_csv('../data/processed/transactions_rules.csv')
model = joblib.load('../models/random_forest_aml_model.pkl')

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,balance_diff,is_high_risk_type,rule_high_value,rule_high_risk_type,rule_account_emptied,rule_amount_spike,rule_velocity,rule_score,rule_based_alert,rule_alert_strict
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,...,9839.64,0,0,0,0,0,0,0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,...,1864.28,0,0,0,0,0,0,0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,...,181.00,1,0,1,1,0,0,2,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,...,181.00,1,0,1,1,0,0,2,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,...,11668.14,0,0,0,0,0,0,0,0,0


In [2]:
features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest',
    'txn_count_sender',
    'total_sent',
    'avg_sent',
    'txn_per_step',
    'amount_deviation',
    'balance_diff',
    'is_high_risk_type',
    'rule_score',
    'rule_based_alert'
]

In [3]:
df['ml_risk_probability'] = model.predict_proba(df[features])[:, 1]

In [4]:
df['ml_risk_score'] = (df['ml_risk_probability'] * 100).round(2)

In [5]:
df['final_risk_score'] = (
    (df['ml_risk_score'] * 0.7) +
    (df['rule_score'] * 6)
).clip(0, 100).round(2)

In [6]:
def assign_risk_band(score):
    if score >= 80:
        return 'Critical'
    elif score >= 60:
        return 'High'
    elif score >= 30:
        return 'Medium'
    else:
        return 'Low'

df['risk_band'] = df['final_risk_score'].apply(assign_risk_band)

In [7]:
alert_queue = df[df['rule_based_alert'] == 1].sort_values(
    by='final_risk_score',
    ascending=False
)

alert_queue.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,rule_account_emptied,rule_amount_spike,rule_velocity,rule_score,rule_based_alert,rule_alert_strict,ml_risk_probability,ml_risk_score,final_risk_score,risk_band
6021364,468,TRANSFER,1165187.89,C1125545468,1165187.89,0.0,C1693214292,0.00,0.00,1,...,1,0,0,3,1,1,1.0,100.0,88.0,Critical
5993690,420,CASH_OUT,430722.59,C1979801086,430722.59,0.0,C207357507,1630593.27,2061315.87,1,...,1,0,0,3,1,1,1.0,100.0,88.0,Critical
5994346,421,TRANSFER,207021.45,C376689274,207021.45,0.0,C1019270751,0.00,0.00,1,...,1,0,0,3,1,1,1.0,100.0,88.0,Critical
3610889,267,CASH_OUT,1654055.02,C1415942962,1654055.02,0.0,C1560476015,2165407.21,3819462.24,1,...,1,0,0,3,1,1,1.0,100.0,88.0,Critical
5994344,421,TRANSFER,267564.83,C787135844,267564.83,0.0,C934789306,0.00,0.00,1,...,1,0,0,3,1,1,1.0,100.0,88.0,Critical


In [8]:
risk_band_summary = df.groupby('risk_band').agg(
    transactions=('isFraud', 'count'),
    fraud_cases=('isFraud', 'sum'),
    fraud_rate=('isFraud', 'mean'),
    avg_risk_score=('final_risk_score', 'mean')
).reset_index()

risk_band_summary

,risk_band,transactions,fraud_cases,fraud_rate,avg_risk_score
0,Critical,8162,7109,8.709875e-01,85.696079
1,High,8381,1049,1.251641e-01,70.243849
2,Low,6327615,3,4.741123e-07,5.643732
3,Medium,18462,52,2.816596e-03,42.607274


In [9]:
alert_summary = alert_queue.groupby('risk_band').agg(
    alerts=('isFraud', 'count'),
    fraud_cases=('isFraud', 'sum'),
    fraud_rate=('isFraud', 'mean'),
    avg_risk_score=('final_risk_score', 'mean')
).reset_index()

alert_summary

,risk_band,alerts,fraud_cases,fraud_rate,avg_risk_score
0,Critical,8162,7109,0.870988,85.696079
1,High,8363,1032,0.123401,70.233301
2,Low,1744851,2,0.000001,14.251725
3,Medium,18045,43,0.002383,42.711383


In [10]:
df.to_csv('../reports/risk_scores.csv', index=False)
alert_queue.to_csv('../reports/prioritised_alert_queue.csv', index=False)
risk_band_summary.to_csv('../reports/risk_band_summary.csv', index=False)
alert_summary.to_csv('../reports/alert_queue_summary.csv', index=False)

## Risk Scoring Summary

This notebook converts model probabilities and rule-based signals into a final AML risk score.

The final risk score combines:
- Machine learning fraud probability
- Rule-based suspicious activity score

Transactions are grouped into Low, Medium, High, and Critical risk bands. The output creates a prioritised alert queue to help investigators focus on the highest-risk transactions first.